In [32]:
# 모델 불러와 적용해보기

import joblib
import pandas as pd


# =========================================
# 1. 저장된 모델 불러오기
# =========================================

model = joblib.load(
    "models/item_cf_model_pc120.pkl"
)

user_item_matrix = model["user_item_matrix"]

item_similarity_df = model["item_similarity_df"]

product_df = model["product_df"]


# =========================================
# 2. 상품 추천 함수
# =========================================

def recommend(user_id, top_n=5):

    # 기존 고객 확인
    
    # 신규 회원 확인
    if user_id not in user_item_matrix.index:
        print("신규 고객입니다.")

        popular_products = (
            purchase_df.groupby("product_id")["quantity"]
            .sum()
            .sort_values(ascending=False)
            .head(top_n)
            .reset_index(name="sales_count")
        )

        return popular_products.merge(
            product_df[["product_id", "category"]],
            on="product_id"
        )[["product_id", "category", "sales_count"]]


    # 여기부터 기존 회원 코드 (수정하지 않음)
    user_ratings = user_item_matrix.loc[user_id]


    purchased = user_ratings[
        user_ratings > 0
    ]

    # 추천 점수 계산
    scores = pd.Series(
        0.0,
        index=item_similarity_df.index
    )

    for product_id, rating in purchased.items():

        scores += (
            item_similarity_df[product_id] * rating
        )

    # 구매한 상품 제외
    scores = scores.drop(
        purchased.index,
        errors="ignore"
    )

    # 추천 점수 순으로 정렬
    scores = scores.sort_values(
        ascending=False
    )

    # 상위 상품 선택
    result = scores.head(top_n).reset_index()

    result.columns = [
        "product_id",
        "recommend_score"
    ]

    # 상품 정보 결합
    result = result.merge(
        product_df,
        on="product_id",
        how="left"
    )

    return result[
        [
            "product_id",
            "category",
            "recommend_score"
        ]
    ]


# =========================================
# 3. 추천 시스템 실행
# =========================================

user_id = "U001"

result = recommend(
    user_id=user_id,
    top_n=5
)

print("\n===== 고객별 상품 추천 결과 =====")

print(result.to_string(index=False))

신규 고객입니다.

===== 고객별 상품 추천 결과 =====
 product_id category  sales_count
         21      CPU          310
        101  MONITOR          301
         31 KEYBOARD          266
        102  MONITOR          250
         22      CPU          242


In [33]:
# 다른 기존 회원에게 추천하기: 아래 숫자를 바꿔 보세요.
print(recommend(int(user_item_matrix.index[5]), top_n=5))

   product_id category  recommend_score
0          56      SSD         0.306083
1          51      SSD         0.286434
2          22      CPU         0.285335
3          93      RAM         0.281781
4          23      CPU         0.274453


In [35]:
# 추천 테스트
print(recommend(2, top_n=5))
print(recommend(5, top_n=5))
print(recommend(11, top_n=5))
print(recommend(9, top_n=5))

신규 고객입니다.
   product_id  category  sales_count
0          21       CPU          310
1         101   MONITOR          301
2          31  KEYBOARD          266
3         102   MONITOR          250
4          22       CPU          242
신규 고객입니다.
   product_id  category  sales_count
0          21       CPU          310
1         101   MONITOR          301
2          31  KEYBOARD          266
3         102   MONITOR          250
4          22       CPU          242
신규 고객입니다.
   product_id  category  sales_count
0          21       CPU          310
1         101   MONITOR          301
2          31  KEYBOARD          266
3         102   MONITOR          250
4          22       CPU          242
   product_id   category  recommend_score
0         104    MONITOR         0.933199
1          14       CASE         0.914003
2          81      POWER         0.908618
3         114  MAINBOARD         0.884507
4          84      POWER         0.797433
